In [2]:
#importing standard packages
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

In [3]:
#importing database
df = pd.read_csv("Assignment2.csv")
df.head()

,date,dc,rmrf,smb,hml,rf,R1,R2,R3,R4,R5,R6,R7,R8,R9,R10
0,1952,1.039208,0.1180,-0.0702,0.0405,0.0166,0.1432,0.0982,0.1471,0.1852,0.1291,0.0931,0.1288,0.1617,0.0508,0.3209
1,1953,0.996975,-0.0105,-0.0080,-0.0870,0.0182,0.0414,0.0282,-0.0124,0.0008,0.0375,-0.0207,-0.0730,-0.0280,-0.0719,-0.1455
2,1954,1.034664,0.4935,-0.0246,0.2574,0.0086,0.5228,0.3877,0.4527,0.4116,0.5855,0.6271,0.5594,0.8194,0.6587,0.7037
3,1955,1.033162,0.2375,-0.0676,0.0571,0.0157,0.3215,0.2087,0.2185,0.2596,0.2059,0.2475,0.1886,0.3581,0.2485,0.1853
4,1956,1.011275,0.0590,-0.0138,-0.0107,0.0246,0.0372,0.0911,0.1006,0.1424,0.2034,0.1000,0.0962,0.0811,0.0529,-0.0093


In [4]:
# 1. Construct excess returns R1e - R10e
portfolios = [f'R{i}' for i in range(1, 11)]
excess_cols = [f'R{i}e' for i in range(1, 11)]

for r, re in zip(portfolios, excess_cols):
    df[re] = df[r] - df['rf']

# 2. Run 10 time-series OLS regressions
factors = ['rmrf', 'smb', 'hml']
X = sm.add_constant(df[factors])

results = {}
alphas = []
betas = []
residuals = []

for col in excess_cols:
    model = sm.OLS(df[col], X).fit()
    results[col] = model
    alphas.append(model.params['const'])
    betas.append(model.params[factors].values)
    residuals.append(model.resid.values)

# 3. Format output into a clean summary table matching the assignment
table_rows = {}
for i, col in enumerate(excess_cols, start=1):
    res = results[col]
    table_rows[f'({i})'] = {
        'Rm-Rf coef': res.params['rmrf'],
        'Rm-Rf (t)':  res.tvalues['rmrf'],
        'SmB coef':   res.params['smb'],
        'SmB (t)':    res.tvalues['smb'],
        'HmL coef':   res.params['hml'],
        'HmL (t)':    res.tvalues['hml'],
        'alpha coef': res.params['const'],
        'alpha (t)':  res.tvalues['const'],
        'R2':         res.rsquared,
        'Obs':        int(res.nobs)
    }

summary_df = pd.DataFrame(table_rows)
print(summary_df.round(3))

               (1)     (2)     (3)     (4)     (5)     (6)     (7)     (8)  \
Rm-Rf coef   1.067   0.940   0.936   0.889   0.931   0.881   0.985   1.061   
Rm-Rf (t)   24.678  25.663  27.318  16.244  16.824  20.744  21.181  23.738   
SmB coef    -0.133  -0.098  -0.031   0.081   0.043   0.018   0.132   0.114   
SmB (t)     -2.199  -1.907  -0.640   1.053   0.554   0.300   2.030   1.824   
HmL coef    -0.448  -0.077   0.051   0.201   0.255   0.421   0.512   0.701   
HmL (t)     -7.581  -1.533   1.088   2.683   3.377   7.255   8.058  11.472   
alpha coef   0.012   0.009   0.005  -0.000   0.000   0.007  -0.018  -0.010   
alpha (t)    1.372   1.263   0.658  -0.032   0.037   0.789  -1.857  -1.139   
R2           0.927   0.923   0.931   0.833   0.841   0.892   0.902   0.923   
Obs         63.000  63.000  63.000  63.000  63.000  63.000  63.000  63.000   

               (9)    (10)  
Rm-Rf coef   1.042   1.187  
Rm-Rf (t)   24.490  20.326  
SmB coef     0.278   0.315  
SmB (t)      4.657   3.84

In [5]:
T = len(df)          # 63 observations
N = len(excess_cols) # 10 test assets
K = len(factors)     # 3 factors

# Convert to numpy arrays
alpha_vec = np.array(alphas).reshape(-1, 1)  # (N, 1)
resid_mat = np.array(residuals).T            # (T, N)
f_mat = df[factors].values                   # (T, K)
f_mean = f_mat.mean(axis=0).reshape(-1, 1)   # (K, 1)

# MLE covariance matrices (ddof=0 divides by T)
Sigma_hat = np.cov(resid_mat, rowvar=False, ddof=0) # (N, N)
Omega_hat = np.cov(f_mat, rowvar=False, ddof=0)     # (K, K)

# GRS statistic
sharpe_term = (f_mean.T @ np.linalg.inv(Omega_hat) @ f_mean).item()
alpha_term  = (alpha_vec.T @ np.linalg.inv(Sigma_hat) @ alpha_vec).item()

grs_stat = ((T - N - K) / N) * (alpha_term / (1 + sharpe_term))
p_value = stats.f.sf(grs_stat, N, T - N - K)  # 1 - CDF of F(10, 50)

print(f"GRS test-statistic: {grs_stat:.4f}")
print(f"p-value:            {p_value:.4f}")

GRS test-statistic: 1.5363
p-value:            0.1545
